In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
# SANITY CHECKS
# beta weight sanity checks
# -> multiply beta weight by 1/binwidth_s and make sure identical to firing rate
"""--------------------------------------------"""

## init

In [ ]:
from sg.models import make_tre, Encoder, StrategyEncoder

encoder = make_tre(Encoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
)
encoder.verify()

encoder_mb = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mb",
)
encoder_mb.verify()

encoder_mf = make_tre(StrategyEncoder, tr_type="dme")(
    subj_id,
    sess_id,
    norm=True,
    stepsize_s=0.1,
    strategy_filter="mf",
)
encoder_mf.verify()

In [ ]:
import numpy as np

c_idxs = np.where(encoder.trial_data.rewarded == 1)[0]
i_idxs = np.where(encoder.trial_data.rewarded == 0)[0]

psths_c = encoder.psths["DLS"][:, c_idxs, :]
psths_i = encoder.psths["DLS"][:, i_idxs, :]

grand_c = psths_c.mean(axis=(0, 1))
grand_i = psths_i.mean(axis=(0, 1))

In [ ]:
plt.figure()
plt.plot(encoder.tbin_centers, grand_c)
plt.plot(encoder.tbin_centers, grand_i)
plt.show()

In [ ]:
encoder.psths["DMS"].shape

In [ ]:
encoder.view_peths(mode="rewarded")

In [ ]:
from core.viz import plot_kdes

idx0, idxm1 = encoder.dm_idxs["block_side_0"], encoder.dm_idxs["block_side_14"]
idx0_r, idxm1_r = encoder.dm_idxs["response_0"], encoder.dm_idxs["response_14"]

plot_kdes(
    {
        "block_side": encoder.encoder_weights[:, idx0 : idxm1 + 1].ravel(),
        "response": encoder.encoder_weights[:, idx0_r : idxm1_r + 1].ravel(),
    }
)

In [ ]:
encoder_mb.view_weights(regr="block_side")

## manifolds

In [ ]:
# compress task variable predicted activity for mb/mf to a principle component (pca or ae) and plot against each other

encoder_mb.fit_encoder()
encoder_mf.fit_encoder()

robs_mb = encoder_mb.encoder.predict(encoder.tvs)
robs_mf = encoder_mf.encoder.predict(encoder.tvs)

In [ ]:
from sklearn.decomposition import PCA

pca_mb = PCA(n_components=1).fit(robs_mb)
pca_mf = PCA(n_components=1).fit(robs_mf)

pc_mb = pca_mb.transform(robs_mb)[:, 0]
pc_mf = pca_mf.transform(robs_mf)[:, 0]

pc_mb_trajs = pc_mb.reshape(encoder.num_trials, encoder.num_bins)
pc_mf_trajs = pc_mf.reshape(encoder.num_trials, encoder.num_bins)

In [ ]:
pca_mb.explained_variance_ratio_, pca_mf.explained_variance_ratio_

In [ ]:
encoder_mb.encoder_weights.shape

In [ ]:
encoder_mb.encoder_weights.shape

In [ ]:
import numpy as np

fig, ax = plt.subplots()

im = ax.imshow(
    encoder_mb.encoder_weights[np.argsort(pca_mb.components_[0])]
    - encoder_mf.encoder_weights[np.argsort(pca_mb.components_[0])],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
fig.colorbar(im)

In [ ]:
encoder_mb.dm_idxs

In [ ]:
import numpy as np

fig, axes = plt.subplots(
    nrows=1, ncols=2, sharey=True, figsize=(3, 2), constrained_layout=True
)

im = axes[0].imshow(
    encoder_mb.encoder_weights[np.argsort(pca_mb.components_[0])],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)
axes[1].imshow(
    encoder_mf.encoder_weights[np.argsort(pca_mb.components_[0])],
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
)

axes[0].set_xlabel("regressor")

axes[0].set_ylabel("units (s.b. loadings)")
axes[1].set_ylabel("units (s.b. loadings)")

axes[0].set_title("mb")
axes[1].set_title("mf")

fig.colorbar(im, ax=axes, shrink=0.8, pad=0.05, label="encoder weight")

plt.show()

In [ ]:
from core.viz import plot_scatter, plot_trajectory
import numpy as np

ax = plot_scatter(
    x=pc_mb,
    y=pc_mf,
    xlabel="mb",
    ylabel="mf",
    cmap="plasma",
    color=np.tile(range(encoder.num_bins), encoder.num_trials),
)

for i in range(encoder.num_trials):
    # s, e = i*encoder.num_bins, (i+1)*encoder.num_bins
    # plot_trajectory(x=pc_mb[s:e], y=pc_mf[s:e], ax=ax)
    plot_trajectory(x=pc_mb_trajs[i], y=pc_mf_trajs[i], ax=ax)

ax.scatter(
    x=pca_mb.components_[0],
    y=np.repeat(-6, encoder.num_units),
    cmap="plasma",
    c=np.argsort(pca_mb.components_[0]),
    alpha=0.5,
    s=0.5,
    marker="|",
)
ax.scatter(
    x=np.repeat(-7, encoder.num_units),
    y=pca_mf.components_[0],
    cmap="plasma",
    c=np.argsort(pca_mf.components_[0]),
    alpha=0.5,
    s=0.5,
    marker="|",
)

In [ ]:
trajs = np.array([pc_mb_trajs, pc_mf_trajs]).transpose(1, 0, 2)
n_unique = np.unique(trajs, axis=0).shape[0]
n_unique_mb = np.unique(pc_mb_trajs, axis=0).shape[0]
n_unique_mf = np.unique(pc_mf_trajs, axis=0).shape[0]

print("coords:", n_unique, "\nmb:", n_unique_mb, "\nmf:", n_unique_mf)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

n_clusters = range(2, trajs.shape[0] // 2)
seeds = range(10)
silhouettes = np.zeros((len(n_clusters), len(seeds)))

trajs_flat = trajs.reshape(trajs.shape[0], -1)

for i, n in enumerate(n_clusters):
    for j, seed in enumerate(seeds):
        km = KMeans(n_clusters=n, random_state=seed)
        cluster_labels = km.fit_predict(trajs_flat)
        silhouettes[i][j] = silhouette_score(trajs_flat, cluster_labels)

n_cluster = n_clusters[np.argmax(silhouettes.mean(axis=1))]
print("cluster w/ best avg. silhouette score: ", n)

In [ ]:
s_mean = silhouettes.mean(axis=1)
s_std = silhouettes.std(axis=1)
plt.figure(figsize=(2, 1), tight_layout=True)
plt.plot(n_clusters, s_mean)
plt.fill_between(n_clusters, s_mean - s_std, s_mean + s_std, alpha=0.5)
plt.show()

In [ ]:
from sklearn.preprocessing import OneHotEncoder as OHE

trial_data_ohe = OHE().fit_transform(encoder.trial_data[encoder.tv_keys]).todense()

np.unique(trial_data_ohe, axis=0).shape

In [ ]:
km = KMeans(n_clusters=n_cluster, random_state=0)
cluster_labels = km.fit_predict(trajs_flat)

fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(6, 4), tight_layout=True)

for i, ax in enumerate(axes.flat):
    idxs = np.where(cluster_labels == i)[0]
    for j in idxs:
        ax = plot_trajectory(x=pc_mb_trajs[j], y=pc_mf_trajs[j], ax=ax)

    ax.set_xlim([-6, 6])
    ax.set_ylim([-6, 6])

    total = np.shape(idxs)[0]
    unique = np.unique(trajs[idxs], axis=0).shape[0]

    ax.set_title(f"cluster {i}, ({total}, {unique})")

In [ ]:
encoder_mb.encoder_weights.shape

In [ ]:
# i would wnat to still look at communication subspace and see the difference in task variables encoding in the private and shared subspace

In [ ]:
np.where(cluster_labels == i)
# are neurons that evlve along the axis for one task variable the same neurons that do something wacky for other task variables?
# plot all the trajectories in dms and dls on the same figure and

## the t-population

In [ ]:
# select the neurons that lie along the axis
# plot their encoding for mb/mf
# is it just a different encoding pattern (i.e., they encode different things)
# or are they silent(er) in another strategy